# Consulta a la base vectorial (Qdrant + multilingual-e5-small)

Este notebook recibe una **pregunta en lenguaje natural** y devuelve una **respuesta estructurada**:

```json
{"question": ..., "answer": "93" | "SIN_EVIDENCIA", "citations": ["D000-0", "D000-1"], "kind": "actual|historico|sin_evidencia", ...}
```

Pipeline:
1. **Parseo** de la pregunta: sitio (`ER-###`), año, objeto (semillas, celdas, ...) e intención (¿pide una cantidad?).
2. **Recuperación densa** en Qdrant (filtrada por sitio si se detecta).
3. **Selección** con reglas de vigencia: registro oficial del año pedido (o vigente = 2048); rumores y archivos sin cifra se descartan.
4. **Citas**: documento en español + su gemelo en Eráviri (mismo sitio, año, fuente y cifra).

Requisito previo: haber corrido `uv run index_corpus.py`.

In [1]:
import json, re, time
from dataclasses import dataclass, field, asdict
from pathlib import Path

import pandas as pd
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

from index_corpus import COLLECTION, DB_PATH, MODEL_NAME, search

ROOT = Path.cwd().parent
CURRENT_YEAR = 2048

model = SentenceTransformer(MODEL_NAME)
client = QdrantClient(path=str(DB_PATH))
print(client.count(COLLECTION).count, "documentos indexados")

/Users/gmeza/PUCP/26-2/NLP/INF657-final-g4/database/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1125 20:13:01.203000 79117 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9085.40it/s]


1500 documentos indexados


## Léxico Eráviri → español
Se usa para traducir los documentos `er` en la evidencia y para reconocer objetos escritos en Eráviri en la pregunta.

In [3]:
lexicon = dict(pd.read_csv(ROOT / "lexico.csv").values)  # er -> es
ITEMS = {"semillas", "celdas", "vehículos", "tablillas", "depósitos"}

def translate(text: str) -> str:
    return " ".join(lexicon.get(tok.strip(".,").lower(), tok) for tok in text.split())

lexicon

{'sava': 'semillas',
 'tavi': 'celdas',
 'nora': 'vehículos',
 'kiri': 'tablillas',
 'mura': 'depósitos',
 'ena': 'disponibles'}

## 1. Parseo de la pregunta

In [6]:
QUANTITY_CUES = re.compile(r"cu[aá]nt|cantidad|inventario|n[uú]mero|registraba|disponibles", re.I)

@dataclass
class ParsedQuery:
    site: str | None
    year: int | None
    item: str | None
    asks_quantity: bool
    kind: str  # actual | historico | sin_evidencia

def parse_query(q: str) -> ParsedQuery:
    site = m.group(0).upper() if (m := re.search(r"ER-\d{3}", q, re.I)) else None
    year = int(m.group(0)) if (m := re.search(r"\b(20\d{2})\b", q)) else None
    words = {lexicon.get(w, w) for w in re.findall(r"\w+", q.lower())}
    item = next(iter(words & ITEMS), None)
    asks_quantity = bool(QUANTITY_CUES.search(q))
    if not asks_quantity:
        kind = "sin_evidencia"
    elif year is not None and year != CURRENT_YEAR:
        kind = "historico"
    else:
        kind = "actual"
    return ParsedQuery(site, year, item, asks_quantity, kind)

parse_query("¿Qué cantidad de semillas registraba ER-070 en 2047?")

ParsedQuery(site='ER-070', year=2047, item='semillas', asks_quantity=True, kind='historico')

## 2–4. Recuperación, selección y salida estructurada

In [7]:
@dataclass
class Evidence:
    doc_id: str
    score: float
    language: str
    year: int
    source: str
    reliability: float
    text: str
    translation: str | None = None

@dataclass
class RAGAnswer:
    question: str
    answer: str
    citations: list[str]
    kind: str
    site: str | None
    year: int | None
    item: str | None
    rationale: str
    evidence: list[Evidence] = field(default_factory=list)
    latency_ms: float = 0.0

    def to_json(self, **kw) -> str:
        return json.dumps(asdict(self), ensure_ascii=False, **kw)

def number_in(text: str) -> str | None:
    # Ignora el número del sitio (ER-###) y devuelve la cifra del inventario.
    m = re.search(r"\b(\d+)\b", re.sub(r"ER-\d+", "", text))
    return m.group(1) if m else None

def answer(question: str, k: int = 10) -> RAGAnswer:
    t0 = time.perf_counter()
    pq = parse_query(question)
    hits = search(client, model, question, k=k, site=pq.site)
    evidence = [
        Evidence(doc_id=p.payload["doc_id"], score=round(p.score, 4), language=p.payload["language"],
                 year=p.payload["year"], source=p.payload["source"], reliability=p.payload["reliability"],
                 text=p.payload["text"],
                 translation=translate(p.payload["text"]) if p.payload["language"] == "er" else None)
        for p in hits
    ]

    def finish(ans, cites, why):
        return RAGAnswer(question, ans, cites, pq.kind, pq.site, pq.year, pq.item, why, evidence,
                         round((time.perf_counter() - t0) * 1000, 1))

    if not pq.asks_quantity:
        return finish("SIN_EVIDENCIA", [], "La pregunta no solicita una cantidad registrada en el corpus.")

    target_year = pq.year or CURRENT_YEAR
    sites = {pq.site} if pq.site else {p.payload["site"] for p in hits}
    candidates = [
        (e, p.payload["site"]) for e, p in zip(evidence, hits)
        if e.source == "oficial" and e.year == target_year and p.payload["site"] in sites
        and number_in(e.text) is not None
        and (pq.item is None or pq.item in (e.translation or e.text))
    ]
    if not candidates:
        return finish("SIN_EVIDENCIA", [], f"No hay registro oficial de {target_year} para la consulta.")

    best, best_site = max(candidates, key=lambda c: (c[0].language == "es", c[0].reliability, c[0].score))
    value = number_in(best.text)
    cites = [e.doc_id for e, s in candidates if s == best_site and number_in(e.text) == value]
    cites.sort(key=lambda d: d != best.doc_id)
    discarded = sorted({e.source for e in evidence if e.source != "oficial"} |
                       ({"registro de otro año"} if any(e.year != target_year for e in evidence) else set()))
    why = f"Registro oficial {target_year} (confiabilidad {best.reliability})"
    if discarded:
        why += f"; se descartan: {', '.join(discarded)}"
    return finish(value, cites, why + ".")

## Ejemplos

In [8]:
for q in [
    "¿Cuántas semillas hay disponibles actualmente en ER-000?",
    "¿Qué cantidad de semillas registraba ER-000 en 2047?",
    "¿Quién dirige ER-000?",
]:
    print(answer(q).to_json(indent=2))
    print()

{
  "question": "¿Cuántas semillas hay disponibles actualmente en ER-000?",
  "answer": "93",
  "citations": [
    "D000-0",
    "D000-1"
  ],
  "kind": "actual",
  "site": "ER-000",
  "year": null,
  "item": "semillas",
  "rationale": "Registro oficial 2048 (confiabilidad 0.95); se descartan: archivo, registro de otro año, rumor.",
  "evidence": [
    {
      "doc_id": "D000-2",
      "score": 0.9181,
      "language": "es",
      "year": 2047,
      "source": "oficial",
      "reliability": 0.9,
      "text": "El registro anterior de ER-000 indica 100 semillas disponibles.",
      "translation": null
    },
    {
      "doc_id": "D000-0",
      "score": 0.9057,
      "language": "es",
      "year": 2048,
      "source": "oficial",
      "reliability": 0.95,
      "text": "El registro oficial de ER-000 confirma 93 semillas disponibles.",
      "translation": null
    },
    {
      "doc_id": "D000-4",
      "score": 0.8974,
      "language": "es",
      "year": 2048,
      "source": "

Vista tabular compacta (sin la evidencia completa):

In [9]:
qs = [
    "Según el registro vigente, ¿cuál es el inventario de ER-013?",
    "¿Cuántas celdas hay disponibles actualmente en ER-001?",
    "¿Qué cantidad de vehículos registraba ER-002 en 2045?",
    "¿Cuántas tavi hay en ER-001?",
]
pd.DataFrame([{k: v for k, v in asdict(answer(q)).items() if k != "evidence"} for q in qs])

,question,answer,citations,kind,site,year,item,rationale,latency_ms
0,"Según el registro vigente, ¿cuál es el inventa...",16,"[D013-0, D013-1]",actual,ER-013,NaN,NaN,Registro oficial 2048 (confiabilidad 0.95); se...,235.1
1,¿Cuántas celdas hay disponibles actualmente en...,26,"[D001-0, D001-1]",actual,ER-001,NaN,celdas,Registro oficial 2048 (confiabilidad 0.95); se...,26.1
2,¿Qué cantidad de vehículos registraba ER-002 e...,SIN_EVIDENCIA,[],historico,ER-002,2045.0,vehículos,No hay registro oficial de 2045 para la consulta.,25.6
3,¿Cuántas tavi hay en ER-001?,26,"[D001-0, D001-1]",actual,ER-001,NaN,celdas,Registro oficial 2048 (confiabilidad 0.95); se...,25.2


## Evaluación

`evaluate_file` ejecuta el pipeline sobre un CSV de preguntas (`qid, question, answer, evidence_ids, kind`). Guarda las predicciones en el formato de entrega (`qid, answer, citations`) y calcula las métricas de `evaluation.py`, más el detalle por tipo, la latencia y la lista de errores.

Los gold se usan **solo** para medir, nunca como contexto de recuperación.

In [10]:
import sys
sys.path.insert(0, str(ROOT))
from evaluation import evaluate

def evidence_hit(gold_ids, cites):
    g = set(str(gold_ids).split("|")) - {"", "nan"}
    return bool(g & set(cites)) if g else not cites

def evaluate_file(path, out_csv):
    gold = pd.read_csv(path)
    results = [answer(q) for q in gold.question]
    pred = pd.DataFrame({"qid": gold.qid,
                         "answer": [r.answer for r in results],
                         "citations": ["|".join(r.citations) for r in results]})
    pred.to_csv(out_csv, index=False)

    metrics = evaluate(pred, gold)
    lat = pd.Series([r.latency_ms for r in results])
    metrics.update(latency_ms_mean=round(lat.mean(), 1), latency_ms_p95=round(lat.quantile(.95), 1), n=len(gold))

    detail = gold.assign(pred_answer=pred.answer, citations=pred.citations,
                         pred_kind=[r.kind for r in results],
                         rationale=[r.rationale for r in results])
    detail["EM"] = detail.answer.astype(str).str.strip().str.lower() == detail.pred_answer.astype(str).str.strip().str.lower()
    detail["evidence_hit"] = [evidence_hit(g, r.citations) for g, r in zip(gold.evidence_ids, results)]
    detail["joint"] = detail.EM & detail.evidence_hit
    by_kind = detail.groupby("kind")[["EM", "evidence_hit", "joint"]].mean().assign(n=detail.groupby("kind").size())
    errors = detail[~detail.joint]
    return metrics, by_kind, errors, detail

### Entrenamiento (`database/train_qa.csv`)

In [11]:
train_metrics, train_by_kind, train_errors, train_detail = evaluate_file(
    "train_qa.csv", "predicciones_train_dense.csv")
print(json.dumps(train_metrics, indent=2, ensure_ascii=False))
train_by_kind

{
  "EM": 1.0,
  "evidence_hit": 1.0,
  "joint": 1.0,
  "by_kind": {
    "actual": 1.0,
    "historico": 1.0,
    "sin_evidencia": 1.0
  },
  "latency_ms_mean": 25.3,
  "latency_ms_p95": 30.0,
  "n": 500
}


,EM,evidence_hit,joint,n
kind,,,,
actual,1.0,1.0,1.0,250
historico,1.0,1.0,1.0,125
sin_evidencia,1.0,1.0,1.0,125


In [12]:
print(f"{len(train_errors)} errores (joint = False)")
train_errors[["qid", "question", "kind", "answer", "pred_answer", "evidence_ids", "citations", "rationale"]]

0 errores (joint = False)


,qid,question,kind,answer,pred_answer,evidence_ids,citations,rationale


### Validación (`validation_qa.csv`)

In [13]:
val_metrics, val_by_kind, val_errors, val_detail = evaluate_file(
    ROOT / "validation_qa.csv", "predicciones_validacion_dense.csv")
print(json.dumps(val_metrics, indent=2, ensure_ascii=False))
val_by_kind

{
  "EM": 1.0,
  "evidence_hit": 1.0,
  "joint": 1.0,
  "by_kind": {
    "actual": 1.0,
    "historico": 1.0,
    "sin_evidencia": 1.0
  },
  "latency_ms_mean": 25.3,
  "latency_ms_p95": 29.1,
  "n": 150
}


,EM,evidence_hit,joint,n
kind,,,,
actual,1.0,1.0,1.0,74
historico,1.0,1.0,1.0,41
sin_evidencia,1.0,1.0,1.0,35


In [14]:
pd.DataFrame({"train": train_metrics, "validation": val_metrics}).drop("by_kind")

,train,validation
EM,1.0,1.0
evidence_hit,1.0,1.0
joint,1.0,1.0
latency_ms_mean,25.3,25.3
latency_ms_p95,30.0,29.1
n,500,150


In [15]:
client.close()